<a href="https://colab.research.google.com/github/mar-olya/compling-Markovich/blob/main/Markovich_fine_tuning_hw%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
!pip install transformers datasets evaluate accelerate -q #доступ к архитектурам гпт / берт и тд; полезно для корпусной лингвистике: можно публиковать исследовательский корпус
!pip install huggingface_hub -q #для авторизации, скачивания моделей

import torch
print(f"GPU доступен: {torch.cuda.is_available()}") #проверяем, работает ли этот процессор
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [2]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# 1. Загрузка датасета
dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [5]:
dataset['test'][0]

{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'label': 2}

In [6]:
# 2. Загрузка модели и токенизатора
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4  # 0 = негативный, 1 = позитивный
).to(device)

id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# 3. Подготовка данных
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir="./results-ag-news",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [9]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
# 6. Обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.196032,0.184059,0.942200
2,0.136998,0.188946,0.948600
3,0.086502,0.219416,0.946200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.15376009945339628, metrics={'train_runtime': 3507.6914, 'train_samples_per_second': 102.632, 'train_steps_per_second': 6.414, 'total_flos': 8558812764910464.0, 'train_loss': 0.15376009945339628, 'epoch': 3.0})

In [11]:
# 7. Оценка
# Оценка на всей тестовой выборке
full_test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"\nТочность на полной тестовой выборке: {full_test_results['eval_accuracy']:.4f}")


Точность на полной тестовой выборке: 0.9505


In [12]:
# Сохраняем модель и токенизатор
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена в папку ./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в папку ./ag_news_model


In [15]:
# 8. Тестирование на новых новостях
from transformers import pipeline

# Маппинг меток (уже определен выше)
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
# Создаем обратный маппинг (на всякий случай)
label2id = {v: k for k, v in id2label.items()}

# Загружаем сохраненную модель
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model",
    device=0 if torch.cuda.is_available() else -1
)

# Три новые новости для тестирования
test_news = [
    "Russia and Ukraine agree to ceasefire negotiations in Istanbul",
    "Manchester United announces new stadium project worth 2 billion pounds",
    "NASA's Perseverance rover discovers organic molecules on Mars"
]

print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ")
print("="*60)

for i, news in enumerate(test_news, 1):
    result = classifier(news)[0]
    category = result['label']  # Теперь это строка, например "World"
    confidence = result['score']

    print(f"\nНовость {i}:")
    print(f"Текст: {news}")
    print(f"Предсказанная категория: {category}")
    print(f"Уверенность: {confidence:.4f}")
    print("-"*40)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ

Новость 1:
Текст: Russia and Ukraine agree to ceasefire negotiations in Istanbul
Предсказанная категория: World
Уверенность: 0.9977
----------------------------------------

Новость 2:
Текст: Manchester United announces new stadium project worth 2 billion pounds
Предсказанная категория: Sports
Уверенность: 0.7510
----------------------------------------

Новость 3:
Текст: NASA's Perseverance rover discovers organic molecules on Mars
Предсказанная категория: Sci/Tech
Уверенность: 0.9921
----------------------------------------
